# ROGII - Wellbore Geology Prediction

**Reference:** 
- [ROGII-Wellbore-Geology-Prediction](https://www.kaggle.com/code/vishwasmishra1234/rogii-wellbore-geology-prediction)
- [XGB Starter - [CV 15]](https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15)

# 1. Imports and configs

In [1]:
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
from koolbox import Trainer
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
import numpy as np

/home/nnmax/Desktop/kaggle/ROGII/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CFG:
    data_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    log_path = Path("/kaggle/working/logs")
    
    seed = 42
    n_splits = 7
    cv = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    is_local = not data_path.exists()
    if is_local:
        data_path = Path("/home/nnmax/Desktop/kaggle/ROGII/data")
        log_path = Path("/home/nnmax/Desktop/kaggle/ROGII/logs")
        log_path.mkdir(parents=True, exist_ok=True)
    device = "gpu"

# 2. Data loading and preprocessing

In [3]:
def recent_mean_diff(values, window):
    values = values[-(window + 1):]
    if len(values) < 2:
        return 0.0
    
    return float(np.diff(values).mean())


def recent_slope(y_values, x_values, window):
    y_values = y_values[-window:]
    x_values = x_values[-window:]
    if len(y_values) < 2:
        return 0.0
    
    centered_x = x_values - x_values.mean()
    denominator = float(np.dot(centered_x, centered_x))
    if denominator == 0.0:
        return 0.0
    
    return float(np.dot(centered_x, y_values - y_values.mean()) / denominator)


def nearest_index(sorted_values, target):
    idx = int(np.searchsorted(sorted_values, target, side='left'))
    if idx >= len(sorted_values):
        return len(sorted_values) - 1
    
    if idx > 0 and abs(sorted_values[idx - 1] - target) <= abs(sorted_values[idx] - target):
        return idx - 1
    
    return idx

 
def fill_and_smooth_gr(values, fallback, radius):
    series = pd.Series(values, dtype='float32').interpolate(limit_direction='both').fillna(fallback)
    if radius <= 0:
        return series.to_numpy(dtype=np.float32)
    
    return series.rolling(radius * 2 + 1, center=True, min_periods=1).mean().to_numpy(dtype=np.float32)


def beam_predict(gr_values, tw_tvt, tw_gr, start_tvt, beam_size, move_cost, emit_scale, radius):
    start_idx = nearest_index(tw_tvt, start_tvt)
    smoothed_gr = fill_and_smooth_gr(gr_values, float(np.nanmean(tw_gr)), radius)

    states = {start_idx: 0.0}
    backpointers = []

    for gr_value in smoothed_gr:
        candidates = {}
        parents = {}
        for idx, cost in states.items():
            for delta in (-1, 0, 1):
                next_idx = idx + delta
                if next_idx < 0 or next_idx >= len(tw_tvt):
                    continue
                
                emit_cost = ((gr_value - tw_gr[next_idx]) ** 2) / emit_scale
                total_cost = cost + emit_cost + move_cost * abs(delta)
                previous = candidates.get(next_idx)
                
                if previous is None or total_cost < previous:
                    candidates[next_idx] = total_cost
                    parents[next_idx] = idx
        
        kept = sorted(candidates.items(), key=lambda item: item[1])[:beam_size]
        states = {idx: cost for idx, cost in kept}
        backpointers.append({idx: parents[idx] for idx, _ in kept})

    final_idx = min(states, key=states.get)
    path = [final_idx]
    for step in range(len(backpointers) - 1, 0, -1):
        path.append(backpointers[step][path[-1]])
    
    path.reverse()
    
    return tw_tvt[np.asarray(path, dtype=np.int32)]


def build_gr_missingness_features(gr_series, mask_start):
    hidden_gr = gr_series.iloc[mask_start:]
    n_hidden = len(hidden_gr)
    is_missing = hidden_gr.isna().to_numpy().astype(np.int8)
    known_gr = gr_series.iloc[:mask_start]

    dist_to_prev = np.full(n_hidden, -1, dtype=np.int32)
    last_valid_idx_in_known = known_gr.last_valid_index()
    last_dist = mask_start - last_valid_idx_in_known if last_valid_idx_in_known is not None else -1
    for i in range(n_hidden):
        if not hidden_gr.isna().iloc[i]:
            dist_to_prev[i] = 0
            last_dist = 0
        elif last_dist >= 0:
            last_dist += 1
            dist_to_prev[i] = last_dist

    dist_to_next = np.full(n_hidden, -1, dtype=np.int32)
    next_dist = -1
    for i in range(n_hidden - 1, -1, -1):
        if not hidden_gr.isna().iloc[i]:
            dist_to_next[i] = 0
            next_dist = 0
        elif next_dist >= 0:
            next_dist += 1
            dist_to_next[i] = next_dist

    run_len = np.zeros(n_hidden, dtype=np.int32)
    run_pos = np.zeros(n_hidden, dtype=np.int32)
    current_len = 0
    for i in range(n_hidden):
        if is_missing[i]:
            run_len[i] = current_len + 1
            run_pos[i] = current_len
            current_len += 1
        else:
            current_len = 0

    gr_valid = (~hidden_gr.isna()).astype(int)
    windows = [11, 51, 101, 301]
    valid_fracs = {}
    for w in windows:
        valid_fracs[f'gr_valid_frac_w{w}'] = (gr_valid.rolling(w, min_periods=1, center=True).sum().to_numpy(dtype=np.float32) / w)

    prefix_gr_missing_rate = np.float32(known_gr.isna().mean())
    cum_missing = is_missing.cumsum(dtype=np.float32)
    hidden_cum_missing_rate = cum_missing / np.maximum(np.arange(1, n_hidden + 1, dtype=np.float32), 1)
    
    return {
        'gr_dist_to_prev_real': dist_to_prev.astype(np.float32),
        'gr_dist_to_next_real': dist_to_next.astype(np.float32),
        'gr_missing_run_len': run_len.astype(np.float32),
        'gr_missing_run_pos': run_pos.astype(np.float32),
        'prefix_gr_missing_rate': np.full(n_hidden, prefix_gr_missing_rate, dtype=np.float32),
        'hidden_cum_missing_rate': hidden_cum_missing_rate.astype(np.float32),
        **valid_fracs,
    }


def build_prefix_residual_features(prefix_residual, prefix_typewell_gr, prefix_gr, known_tvt, known_md, windows):
    feats = {}
    for w in windows:
        resid_w = prefix_residual[-w:] if len(prefix_residual) >= w else prefix_residual
        feats[f'prefix_resid_mean_{w}'] = np.float32(np.mean(resid_w))
        feats[f'prefix_resid_std_{w}'] = np.float32(np.std(resid_w))
        feats[f'prefix_resid_median_{w}'] = np.float32(np.median(resid_w))
        feats[f'prefix_resid_iqr_{w}'] = np.float32(np.percentile(resid_w, 75) - np.percentile(resid_w, 25))
        resid_tvt = prefix_residual[-w:] if len(prefix_residual) >= w else prefix_residual
        tvt_w = known_tvt[-w:] if len(known_tvt) >= w else known_tvt
        md_w = known_md[-w:] if len(known_md) >= w else known_md
        feats[f'prefix_resid_tvt_slope_{w}'] = np.float32(recent_slope(resid_tvt, tvt_w, w))
        feats[f'prefix_resid_md_slope_{w}'] = np.float32(recent_slope(resid_tvt, md_w, w))
        gr_w = prefix_gr[-w:] if len(prefix_gr) >= w else prefix_gr
        tw_w = prefix_typewell_gr[-w:] if len(prefix_typewell_gr) >= w else prefix_typewell_gr
        if len(gr_w) >= 2 and np.std(gr_w) > 0 and np.std(tw_w) > 0:
            corr = np.corrcoef(gr_w, tw_w)[0, 1]
            feats[f'prefix_gr_tw_corr_{w}'] = np.float32(corr if not np.isnan(corr) else 0.0)
        else:
            feats[f'prefix_gr_tw_corr_{w}'] = np.float32(0.0)
    return feats




def build_typewell_geology_features(tw_geology, tw_tvt, last_known_tvt, n_hidden, offsets):
    geology_labels = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'LTHL', 'LTGT', 'LBHL', 'MNSS', 'BUDA']
    feats = {}
    for offset in offsets:
        target_tvt = last_known_tvt + float(offset)
        idx = nearest_index(tw_tvt, target_tvt)
        geol = tw_geology[idx] if idx < len(tw_geology) else None
        for label in geology_labels:
            feats[f'geology_{label}_offset_{int(offset)}'] = np.full(n_hidden, np.int8(1 if geol == label else 0), dtype=np.int8)
        feats[f'geology_null_offset_{int(offset)}'] = np.full(n_hidden, np.int8(1 if pd.isna(geol) else 0), dtype=np.int8)
    return feats


def build_hidden_features(horizontal_path, typewell_path, is_train):
    well = horizontal_path.name.split('__')[0]
    df = pd.read_csv(horizontal_path)
    mask = df['TVT_input'].isna().to_numpy()
    if not mask.any():
        return None

    mask_start = int(np.flatnonzero(mask)[0])
    if mask_start == 0:
        return None

    known = df.iloc[:mask_start].copy()
    hidden = df.iloc[mask_start:].copy()
    last_known = known.iloc[-1]

    tw = pd.read_csv(typewell_path)
    tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
    tw_gr = tw['GR'].to_numpy(dtype=np.float32)
    tw_geology = tw['Geology'].to_numpy() if 'Geology' in tw.columns else np.full(len(tw), np.nan, dtype=object)

    gr_full = df['GR'].interpolate(limit_direction='both')
    if gr_full.isna().any():
        gr_full = gr_full.fillna(float(np.nanmean(tw_gr)))

    gr_roll5 = gr_full.rolling(5, center=True, min_periods=1).mean()
    gr_roll21 = gr_full.rolling(21, center=True, min_periods=1).mean()
    gr_roll50 = gr_full.rolling(50, center=True, min_periods=1).mean()
    gr_grad = gr_full.diff().fillna(0.0)

    gr_std5 = gr_full.rolling(5, center=True, min_periods=1).std().fillna(0.0)
    gr_std21 = gr_full.rolling(21, center=True, min_periods=1).std().fillna(0.0)
    gr_lag1 = gr_full.shift(1).bfill()
    gr_lead1 = gr_full.shift(-1).ffill()
    gr_lag5 = gr_full.shift(5).bfill()
    gr_lead5 = gr_full.shift(-5).ffill()
    gr_cumsum = gr_full.cumsum()

    known_tvt = known['TVT_input'].to_numpy(dtype=np.float32)
    known_md = known['MD'].to_numpy(dtype=np.float32)
    known_z = known['Z'].to_numpy(dtype=np.float32)

    prefix_typewell_gr = np.interp(known_tvt, tw_tvt, tw_gr)
    prefix_gr = gr_full.iloc[:mask_start].to_numpy(dtype=np.float32)
    prefix_residual = prefix_gr - prefix_typewell_gr
    prefix_tw_rmse = float(np.sqrt(np.mean(prefix_residual ** 2)))
    prefix_tw_mae = float(np.mean(np.abs(prefix_residual)))

    resid_windows = [50, 100, 300]
    prefix_resid_feats = build_prefix_residual_features(prefix_residual, prefix_typewell_gr, prefix_gr, known_tvt, known_md, resid_windows)
    p2_offsets = np.array([-60, -40, -25, -15, -10, -5, 0, 5, 10, 15, 25, 40, 60], dtype=np.float32)

    last_known_tvt = float(last_known['TVT_input'])
    hidden_gr = hidden['GR'].to_numpy(dtype=np.float32)
    beam_cons = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 30, 20.0, 144.0, 2)
    beam_loose = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 30, 8.0, 64.0, 2)

    hidden_gr_filled = gr_full.iloc[mask_start:].to_numpy(dtype=np.float32)
    offset_diffs = {
        f'tw_diff_{int(offset)}': hidden_gr_filled - np.float32(np.interp(last_known_tvt + float(offset), tw_tvt, tw_gr))
        for offset in np.array([-60, -40, -25, -15, -10, -5, 0, 5, 10, 15, 25, 40, 60], dtype=np.float32)
    }

    frac_hidden = ((hidden.index - mask_start) / max(len(hidden) - 1, 1)).astype(np.float32)
    gr_miss_feats = build_gr_missingness_features(df['GR'], mask_start)

    features = pd.DataFrame({
        'well': well,
        'prediction_id': [f'{well}_{row_idx}' for row_idx in hidden.index],
        'row_idx': hidden.index.to_numpy(dtype=np.int32),
        'last_known_tvt': np.float32(last_known_tvt),
        'known_len': np.int32(mask_start),
        'hidden_len': np.int32(len(hidden)),
        'frac_hidden': frac_hidden,
        'md': hidden['MD'].to_numpy(dtype=np.float32),
        'z': hidden['Z'].to_numpy(dtype=np.float32),
        'x': hidden['X'].to_numpy(dtype=np.float32),
        'y': hidden['Y'].to_numpy(dtype=np.float32),
        'gr': hidden_gr_filled,
        'gr_missing': hidden['GR'].isna().to_numpy(dtype=np.int8),
        'gr_roll5': gr_roll5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_roll21': gr_roll21.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_roll50': gr_roll50.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_grad': gr_grad.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_std5': gr_std5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_std21': gr_std21.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lag1': gr_lag1.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lead1': gr_lead1.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lag5': gr_lag5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lead5': gr_lead5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_cumsum': (gr_cumsum.iloc[mask_start:] - gr_cumsum.iloc[mask_start - 1]).to_numpy(dtype=np.float32) if mask_start > 0 else gr_cumsum.iloc[mask_start:].to_numpy(dtype=np.float32),
        'dmd': (hidden['MD'] - float(last_known['MD'])).to_numpy(dtype=np.float32),
        'dz': (hidden['Z'] - float(last_known['Z'])).to_numpy(dtype=np.float32),
        'dx': (hidden['X'] - float(last_known['X'])).to_numpy(dtype=np.float32),
        'dy': (hidden['Y'] - float(last_known['Y'])).to_numpy(dtype=np.float32),
        'dx_dmd': ((hidden['X'] - float(last_known['X'])) / np.maximum(hidden['MD'] - float(last_known['MD']), 1e-5)).to_numpy(dtype=np.float32),
        'dy_dmd': ((hidden['Y'] - float(last_known['Y'])) / np.maximum(hidden['MD'] - float(last_known['MD']), 1e-5)).to_numpy(dtype=np.float32),
        'dz_dmd': ((hidden['Z'] - float(last_known['Z'])) / np.maximum(hidden['MD'] - float(last_known['MD']), 1e-5)).to_numpy(dtype=np.float32),
        'dist_xy': np.sqrt((hidden['X'] - float(last_known['X'])) ** 2 + (hidden['Y'] - float(last_known['Y'])) ** 2).to_numpy(dtype=np.float32),
        'dist_xyz': np.sqrt((hidden['X'] - float(last_known['X'])) ** 2 + (hidden['Y'] - float(last_known['Y'])) ** 2 + (hidden['Z'] - float(last_known['Z'])) ** 2).to_numpy(dtype=np.float32),
        'prefix_tvt_step20': np.float32(recent_mean_diff(known_tvt, 20)),
        'prefix_tvt_step100': np.float32(recent_mean_diff(known_tvt, 100)),
        'prefix_tvt_md_slope100': np.float32(recent_slope(known_tvt, known_md, 100)),
        'prefix_tvt_z_slope100': np.float32(recent_slope(known_tvt, known_z, 100)),
        'prefix_tw_rmse': np.float32(prefix_tw_rmse),
        'prefix_tw_mae': np.float32(prefix_tw_mae),
        'beam_cons_delta': (beam_cons - np.float32(last_known_tvt)).astype(np.float32),
        'beam_loose_delta': (beam_loose - np.float32(last_known_tvt)).astype(np.float32),
        'beam_gap': (beam_loose - beam_cons).astype(np.float32),
    })

    for name, values in offset_diffs.items():
        features[name] = values.astype(np.float32)

    for name, values in gr_miss_feats.items():
        features[name] = values

    for name, values in prefix_resid_feats.items():
        features[name] = np.full(len(hidden), values, dtype=np.float32) if np.ndim(values) == 0 else values

        features[name] = values



    geology_feats = build_typewell_geology_features(tw_geology, tw_tvt, last_known_tvt, len(hidden), p2_offsets)
    for name, values in geology_feats.items():
        features[name] = values

    fh = frac_hidden.to_numpy(dtype=np.float32)
    gr_missing_arr = hidden['GR'].isna().to_numpy(dtype=np.float32)
    features['gr_missing_x_frac_hidden'] = (gr_missing_arr * fh).astype(np.float32)
    features['gr_dist_to_prev_real_x_frac_hidden'] = (gr_miss_feats['gr_dist_to_prev_real'] * fh).astype(np.float32)
    features['gr_missing_run_len_x_frac_hidden'] = (gr_miss_feats['gr_missing_run_len'] * fh).astype(np.float32)
    features['hidden_cum_missing_rate_x_frac_hidden'] = (gr_miss_feats['hidden_cum_missing_rate'] * fh).astype(np.float32)

    if is_train:
        features['target'] = (hidden['TVT'].to_numpy(dtype=np.float32) - np.float32(last_known_tvt)).astype(np.float32)

    return features


def build_dataset(path, is_train):
    parts = []
    
    paths = sorted(path.glob('*__horizontal_well.csv'))
    for horizontal_path in tqdm(paths, desc=f"Building {path.name}"):
        well = horizontal_path.name.split('__')[0]
        typewell_path = path / f'{well}__typewell.csv'
        if not typewell_path.exists():
            continue
        
        features = build_hidden_features(horizontal_path, typewell_path, is_train=is_train)
        if features is not None:
            parts.append(features)

    return pd.concat(parts, ignore_index=True)

In [ ]:
train_df = build_dataset(CFG.data_path / 'train', is_train=True)
test_df = build_dataset(CFG.data_path / 'test', is_train=False)

features = [column for column in train_df.columns if column not in {'well', 'prediction_id', 'target'}]

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"#features:   {len(features)}")

Building train:   0%|          | 0/773 [00:00<?, ?it/s]/tmp/ipykernel_69505/576643112.py:302: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[name] = values
/tmp/ipykernel_69505/576643112.py:306: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features['gr_missing_x_frac_hidden'] = (gr_missing_arr * fh).astype(np.float32)
/tmp/ipykernel_69505/576643112.py:307: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

Train shape: (3783989, 234)
Test shape:  (14151, 233)
#features:   231


: 

# 3. Training

In [ ]:
model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective='regression',
    random_state=CFG.seed,
    force_row_wise=True,
    device=CFG.device,
    verbosity=-1,
    n_jobs=-1,
)

trainer = Trainer(
    estimator=model,
    cv=CFG.cv,
    metric=root_mean_squared_error,
    task="regression",
    verbose=True,
    cv_args={"groups": train_df["well"]},
    save=True,
    save_path=CFG.log_path
)

trainer.fit(
    train_df[features], 
    train_df["target"]
)

Training LGBMRegressor



1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


--- Fold 0 - root_mean_squared_error: 15.1767 - Time: 262.02 s


# 4. Inference

In [ ]:
test_preds = trainer.predict(test_df[features])
test_preds = test_preds + test_df['last_known_tvt']

In [ ]:
submission = pd.DataFrame({
    'id': test_df['prediction_id'],
    'tvt': test_preds,
})

submission.to_csv("submission.csv", index=False)
submission

,id,tvt
0,000d7d20_1442,11747.859817
1,000d7d20_1443,11747.867227
2,000d7d20_1444,11747.894631
3,000d7d20_1445,11747.356012
4,000d7d20_1446,11747.349334
...,...,...
14146,00e12e8b_6379,11598.858156
14147,00e12e8b_6380,11598.864429
14148,00e12e8b_6381,11598.857002
14149,00e12e8b_6382,11598.912012
